In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
data_licences = pd.read_parquet("data/data_licences/data_licences.parquet")
gdf_dep = gpd.read_file("departements.geojson")
data_pop = pd.read_csv("data/data_population/population_dept.csv")

In [ ]:
#On ne garde que les données de population totale pour 2016 et 2022, et on ordonne les départements
data_pop_clean = data_pop[data_pop["type_mesure"] == "PTOT"]
data_pop_clean = data_pop_clean[(data_pop_clean["annee"] == 2016) | (data_pop_clean["annee"] == 2022)]
data_pop_clean = data_pop_clean.sort_values(by="code_dep", ascending=True).reset_index()

# On retire les départements non cartographiables et on crée une copie
data_licences_clean = data_licences.dropna(subset=["code_dep"]).copy()
data_pop_clean = data_pop_clean.dropna(subset=["code_dep"]).copy()

# Conversion en string
data_licences_clean["code_dep"] = data_licences_clean["code_dep"].astype(str)
data_pop_clean["code_dep"] = data_pop_clean["code_dep"].astype(str)

In [ ]:
def plot_licences(annee, sport='all', title=None):
    # filtrage sport
    if sport != 'all':
        df_filtered = data_licences_clean[data_licences_clean["code_sport"] == sport]
    else:
        df_filtered = data_licences_clean.copy()

    # agrégation
    df_agg_lic = df_filtered[df_filtered["annee"] == annee].groupby("code_dep")["licences_annuelles"].sum().reset_index()

    #sélection de l'année pour la population de référence
    if annee in [2016, 2017, 2018, 2019, 2020, 2021]:
        df_pop = data_pop_clean[data_pop_clean["annee"] == 2016].copy()
    elif annee in [2022, 2023, 2024]:
        df_pop = data_pop_clean[data_pop_clean["annee"] == 2022].copy()

    # fusion avec la géométrie
    gdf_plot = gdf_dep.merge(df_agg_lic, left_on="code", right_on="code_dep", how="left")
    gdf_plot = gdf_plot.merge(df_pop[["population","code_dep"]], left_on="code", right_on="code_dep", how="left")

    #création de la variable de proportion de licenciés par département 
    gdf_plot["licences_annuelles_relatives"] = gdf_plot["licences_annuelles"]/gdf_plot["population"]

    # figure
    fig, ax = plt.subplots(figsize=(10, 12))

    gdf_plot.plot(
        column="licences_annuelles_relatives",
        ax=ax,
        legend=True,
        cmap="OrRd",
        edgecolor="grey",
        linewidth=0.5,
        missing_kwds={
            "color": "lightgrey",
            "edgecolor": "grey",
            "hatch": "//",
            "label": "Données manquantes"
        }
    )

    ax.set_title(title if title else f"Proportion de licenciés ({'tous les sports' if sport=='all' else sport}) par département ({annee})", fontsize=16)
    ax.set_axis_off()
    plt.show()


In [ ]:
plot_licences(2024, sport='HAN')

In [ ]:
# GeoDataFrame des départements
gdf = gpd.read_file("./departements.geojson")  # adapte le chemin si nécessaire
gdf["code"] = gdf["code"].astype(str)  # s'assurer que c'est string

# On remplace les NaN dans Code_sport par "Unknown"
data_licences_clean["code_sport"] = data_licences_clean["code_sport"].fillna("Unknown")

# Création des menus déroulants
annee = sorted(data_licences_clean["annee"].unique())
codes_sports = sorted(data_licences_clean["code_sport"].unique())

annee_widget = widgets.Dropdown(options= annee, description="Année :", value=2016)
sport_widget = widgets.Dropdown(options=["all"] + codes_sports, description="Sport:", value="all")

# Fonction pour filtrer et agréger les licences
def filter_aggregate_pop(df, year, sport):
    df_filtered = df.copy()
    df_filtered = df_filtered[df_filtered["annee"] == year]
    if year in [2016, 2017, 2018, 2019, 2020, 2021]:
        df_pop = data_pop_clean[data_pop_clean["annee"] == 2016].copy()
    elif year in [2022, 2023, 2024]:
        df_pop = data_pop_clean[data_pop_clean["annee"] == 2022].copy()
    
    if sport != "all":
        df_filtered = df_filtered[df_filtered["code_sport"] == sport]
  
    df_filtered_aggregated = df_filtered.groupby("code_dep")["licences_annuelles"].sum().reset_index()    
    df_filtered_with_pop = df_filtered_aggregated.merge(df_pop[["population", "code_dep"]], left_on="code_dep", right_on="code_dep", how="left")
    df_filtered_with_pop["licences_annuelles_relatives"] = df_filtered_with_pop["licences_annuelles"]/df_filtered_with_pop["population"]

    return df_filtered_with_pop

# Fonction pour tracer la carte
def plot_map(df_agg, title="Licences par département"):
    # Merge avec le GeoDataFrame
    gdf_plot = gdf.merge(df_agg, left_on="code", right_on="code_dep", how="left")
    gdf_plot["licences_annuelles_relatives"] = gdf_plot["licences_annuelles_relatives"].fillna(0)
    
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    gdf_plot.plot(column="licences_annuelles_relatives", cmap="OrRd", linewidth=0.5, edgecolor="gray",
                  legend=True, ax=ax, norm=Normalize(vmin=0, vmax=gdf_plot["licences_annuelles"].max()))
    ax.set_title(title, fontsize=16)
    ax.axis("off")
    plt.show()

# Callback pour mettre à jour les cartes
def update_maps(change=None):
    clear_output(wait=True)
    display(annee_widget, sport_widget)
    
    df1 = filter_aggregate_pop(data_licences_clean, annee_widget.value, sport_widget.value)
    
    plot_map(
        df1,
        title=f"Proportion de licenciés ({'tous les sports' if sport_widget.value=='all' else sport_widget.value}) "
              f"par département ({annee_widget.value})"
    )

# Liaison des widgets à la fonction de callback
annee_widget.observe(update_maps, names='value')
sport_widget.observe(update_maps, names='value')

# Affichage initial
display(annee_widget, sport_widget)
update_maps()
